In [11]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [12]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC4904/FORS2.2022-02-07T08_36_57.209/NGC-4904_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC3141/UGC-3141_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     371   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        52   (2094, 964)   float32   


In [13]:
image_cut = image[49:149, 0:1890]
image_error_cut = image_error[49:149, 0:1890]

In [14]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.00394
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [15]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [16]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC4904/FORS2.2022-02-07T08_36_57.209/PSF/Real_seeing_NGC4904.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [17]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, a, b):
    func = model_convolved(x, params1, params2, params3)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5 a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choc_area = single_integral(x, params5, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area+ choc_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area, (choc_area*100)/sum_area     

## Halpha

In [18]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC4904/FORS2.2022-02-07T08_36_57.209/Fit/Halpha_fit.csv", index_col=0)

In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)
data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

9.364961469455304
9.873318110712141
1.8235596546104387
1.8174670648003044
12.08305314436962
13.03246541396462
           Peak 1     Peak 2        Peak 3
Blue    99.638360  16.075831  3.707607e-01
Green    0.000003  79.319350  1.223677e-10
Purple   0.361638   4.604819  9.962924e+01


In [20]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

19.999688304979024
20.954999894210033
3.8101465580082965
3.8032672716663423
25.14531082382416
26.933460289553956
           Peak 1     Peak 2        Peak 3
Blue    99.582886  19.998029  4.530099e-01
Green    0.000003  74.601526  2.244581e-10
Purple   0.417110   5.400445  9.954699e+01


In [21]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

26.36686978424539
27.375460633590517
4.894779238551386
4.879161935045898
32.23623107372124
34.12919014334055
           Peak 1     Peak 2        Peak 3
Blue    99.484401  24.341051  5.577326e-01
Green    0.000008  68.747748  3.042074e-10
Purple   0.515591   6.911201  9.944227e+01


## HBeta

In [22]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC3141/Fit/Hbeta_fit.csv", index_col = 0)

In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

6.382130347306047
6.806717708626221
0.3377079792913431
0.6388298179358083
4.616706743970201
5.554635230464923
           Peak 1     Peak 2     Peak 3
Blue    99.649207  62.110822   0.644445
Green    0.000948  23.424755   0.000197
Purple   0.349846  14.464423  99.355358


In [24]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

13.778888055326666
14.571632403533524
0.8226824763907302
1.3856806379292435
9.767358031329842
11.515373260581741
           Peak 1     Peak 2     Peak 3
Blue    99.598990  66.340231   0.775227
Green    0.001049  19.074690   0.000244
Purple   0.399961  14.585079  99.224529


In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


18.3025991779408
19.146949601104573
1.2333442057820978
1.8456247046342384
12.61826045437713
14.503329746366939
           Peak 1     Peak 2     Peak 3
Blue    99.509316  69.157995   0.946539
Green    0.001390  14.908923   0.000294
Purple   0.489294  15.933082  99.053167


## NII

In [26]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC3141/Fit/NII_fit.csv", index_col = 0)

In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

10.583861558307007
11.133500544194778
0.8233127714564707
0.7815812041999252
10.467552334572792
11.943452525305185
           Peak 1     Peak 2     Peak 3
Blue    99.774987  36.420484   0.379939
Green    0.002928  56.368006   0.000334
Purple   0.222085   7.211510  99.619727


In [28]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

22.628223191958767
23.660538547671877
1.81340594299314
1.7406594577084222
21.729003054952774
24.4881860460082
           Peak 1    Peak 2     Peak 3
Blue    99.740940  43.09298   0.466101
Green    0.003258  48.88739   0.000425
Purple   0.255802   8.01963  99.533474


In [29]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

29.82852718533289
30.917841856923907
2.505257785155583
2.4215629811976944
27.63905400030286
30.595874099058896
           Peak 1     Peak 2     Peak 3
Blue    99.679235  48.722086   0.577762
Green    0.004414  41.727229   0.000519
Purple   0.316351   9.550684  99.421719


## SII

In [30]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC3141/Fit/SII_fit.csv", index_col = 0)

In [31]:
fit

,Component 1,Component 2,Component 3
I_e,0.0803,0.0001,0.0446
r_e,31.8857,219.5423,29.9799
n,3.1163,5.7049,3.8440
x_0,24.7762,48.5641,91.7309


In [32]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

9.52055609702465
9.931514121602781
0.5007733633225031
0.5868565053724563
8.03117123393764
9.157586482934342
           Peak 1     Peak 2     Peak 3
Blue    99.752536  53.275433   0.406147
Green    0.076371  37.935007   0.038618
Purple   0.171093   8.789561  99.555236


In [33]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

20.407571024676514
21.18043821540391
1.2168417727874952
1.3788860421151345
16.861597719870524
18.96532534333973
           Peak 1     Peak 2     Peak 3
Blue    99.716659  57.214580   0.493104
Green    0.087004  33.936509   0.046845
Purple   0.196338   8.848911  99.460052


In [34]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

26.986692185091695
27.798341896127017
1.8023084489633268
1.9780145126382567
21.62126505019571
23.87870404503387
           Peak 1     Peak 2     Peak 3
Blue    99.648382  60.303755   0.605942
Green    0.109184  29.846704   0.057635
Purple   0.242434   9.849541  99.336424


## OIII

In [35]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC3141/Fit/OIII_fit.csv", index_col = 0)

In [36]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

7.637133870414488
7.5498271063340265
0.4126143880811364
0.41277694328517184
5.035882845924625
5.660708171926587
           Peak 1     Peak 2     Peak 3
Blue    96.429135  33.296346   0.144625
Green    3.563196  65.951657   5.403682
Purple   0.007670   0.751997  94.451693


In [37]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

16.35892756039667
16.210226684812866
1.0192691653505745
1.0196572560140522
10.652982697530163
11.81965005191428
           Peak 1     Peak 2     Peak 3
Blue    95.832617  32.455403   0.175773
Green    4.158678  66.745379   6.386080
Purple   0.008705   0.799218  93.438147


In [38]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

21.76409612665711
21.578106200916913
1.6669508706206546
1.6676295601140787
13.678185087335377
14.928316792456215
           Peak 1     Peak 2     Peak 3
Blue    94.987733  33.941099   0.214593
Green    5.001375  65.299111   7.957810
Purple   0.010892   0.759790  91.827598
